# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each entity in the Croissant schema (record set, field, column) is referenced by its `@id` value.

In [ ]:
# List available record sets and fields by their @id
record_sets = list(dataset.record_sets)
print("Record sets found:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  name: {rs.get('name','')}")
    print(f"  description: {rs.get('description','')}")
    print("  Fields:")
    for field in rs.get('field', []):
        print(f"    - @id: {field['@id']} name: {field.get('name','')} (dataType: {field.get('dataType','')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract records using their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set {record_set_id}: columns={df.columns.tolist()}")
            print(df.head(2), "\n")
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for record set {record_set_id}: {e}")

# For further analysis, select the first available record set
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"Selected record set for EDA: {selected_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We reference columns/fields using their `@id`.

In [ ]:
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Find a numeric field by @id (e.g., GAD-7 total score, named 'gad_7_score')
    numeric_field_id = None
    group_field_id = None
    # Try to identify numeric and grouping fields automatically
    for col in df.columns:
        if col.lower().startswith('gad') or col.lower().startswith('phq') or 'score' in col.lower():
            numeric_field_id = col
        if col.lower() in ['gender', 'village', 'marital_status', 'level_of_education']:
            group_field_id = col
        if numeric_field_id and group_field_id:
            break
    # If not found, fallback
    if not numeric_field_id:
        numeric_field_id = df.select_dtypes('number').columns[0] if len(df.select_dtypes('number').columns) else df.columns[0]
    if not group_field_id:
        group_field_id = 'gender' if 'gender' in df.columns else df.columns[0]

    print(f"Using numeric field '@id': {numeric_field_id}")
    print(f"Using group field '@id': {group_field_id}")

    threshold = 10
    # Filtering records
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field_id} not found in DataFrame.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and the group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_field_id in df.columns:
    # Histogram of numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Groupwise bar plot
    if group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=group_means)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Kilifi dataset provides detailed survey information on mental health indicators and demographics from Kilifi County, Kenya, and is accessible using the Croissant standard.
- Using `mlcroissant`, we loaded metadata, enumerated record sets and fields, and extracted tabular records for analysis.
- Exploratory analyses demonstrate basic filtering, normalization, and group mean comparisons for numeric mental health scoring fields.
- Data visualizations illustrate overall score distributions and differences between demographic groups (e.g., by gender or education).

**For deeper insights, consider further examining missing values, more complex groupings, and integrating metadata for interpretability.**